In [ ]:
# Read API credentials from environment variables
import os

CLIENT_KEY = os.getenv("TIKTOK_CLIENT_KEY")
CLIENT_SECRET = os.getenv("TIKTOK_CLIENT_SECRET")

print("Credentials loaded")

In [ ]:
import requests

TOKEN_URL = "https://open.tiktokapis.com/v2/oauth/token/"

def get_access_token():
    payload = {
        "client_key": CLIENT_KEY,
        "client_secret": CLIENT_SECRET,
        "grant_type": "client_credentials"
    }

    headers = {
        "Content-Type": "application/x-www-form-urlencoded"
    }

    resp = requests.post(TOKEN_URL, data=payload, headers=headers, timeout=30)

    if resp.status_code != 200:
        print("Token request failed:", resp.text)
        return None

    token = resp.json().get("access_token")
    print("Access token received")

    return token

token = get_access_token()
print("Token:", token)

In [ ]:
# Query TikTok Commercial Content API

API_URL = "https://open.tiktokapis.com/v2/research/adlib/commercial_content/query/?fields=id,create_date,brand_names,creator,videos"

def fetch_tiktok_commercial_data():

    access_token = get_access_token()

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    payload = {
        "filters": {
            "content_published_date_range": {
                "min": "20240101",
                "max": "20241231"
            },
            "creator_country_code": "FR"
        },
        "max_count": 10
    }

    resp = requests.post(API_URL, json=payload, headers=headers, timeout=30)

    print("Status Code:", resp.status_code)

    if resp.status_code != 200:
        print(resp.text)
        return None

    return resp.json()

data = fetch_tiktok_commercial_data()
print(data)

In [ ]:
# Display simplified results

def display_results(data):

    contents = data["data"]["commercial_contents"]

    print("\nTikTok Commercial Content Results\n")

    for item in contents:

        creator = item["creator"]["username"]
        date = item["create_date"]

        brand = "None"
        if item["brand_names"]:
            brand = item["brand_names"][0]

        video_url = item["videos"][0]["url"]

        print("Creator:", creator)
        print("Date:", date)
        print("Brand:", brand)
        print("Video:", video_url)
        print("----------------------------------------")

display_results(data)